In [1]:

import os
import cv2
import numpy as np
import librosa
import soundfile as sf
import moviepy.editor as mp
import torch
import torchaudio
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image


In [2]:
# Base dataset directory
DATASET_BASE = Path("datasets")

# Output directory for preprocessed data (you can create inside your framework folder)
OUTPUT_BASE = Path("preprocessed_data")
OUTPUT_BASE.mkdir(exist_ok=True)

# Subfolders
FACEFORENSICS_PATH = DATASET_BASE / "FaceForensics++"
CELEBDF_PATH = DATASET_BASE / "CelebDF-v2"
ASVSPOOF_PATH = DATASET_BASE / "ASVspoof"
FAKEAVCELEB_PATH = DATASET_BASE / "FakeAVCeleb"   # (wait for it, just define now)


In [2]:
print("✅ Environment Check")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"OpenCV: {cv2.__version__}")
print(f"Librosa: {librosa.__version__}")


✅ Environment Check
PyTorch: 2.5.1+cu121
CUDA Available: True
GPU Name: NVIDIA RTX A4000
OpenCV: 4.12.0
Librosa: 0.11.0


In [3]:
def ensure_dir(path):
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)

def list_files_with_ext(base_path, extensions):
    """Return list of all files matching given extensions."""
    files = []
    for ext in extensions:
        files.extend(Path(base_path).rglob(f"*{ext}"))
    return files


In [4]:


IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp"]
VIDEO_EXTS = [".mp4", ".avi", ".mov", ".mkv"]
AUDIO_EXTS = [".wav", ".flac", ".mp3"]
def detect_input_type(file_path):
    ext = file_path.suffix.lower()
    if ext in IMAGE_EXTS:
        return "image"
    elif ext in VIDEO_EXTS:
        return "video"
    elif ext in AUDIO_EXTS:
        return "audio"
    else:
        return "unknown"



In [5]:
def scan_datasets(base_dir):
    dataset_summary = {}

    for dataset_name in os.listdir(base_dir):
        dataset_path = Path(base_dir) / dataset_name
        if not dataset_path.is_dir():
            continue

        print(f"\n📁 Scanning: {dataset_name}")
        counts = {"image": 0, "video": 0, "audio": 0}

        for f in dataset_path.rglob("*.*"):
            file_type = detect_input_type(f)
            if file_type in counts:
                counts[file_type] += 1

        dataset_summary[dataset_name] = counts
        print(f"  🖼️ Images: {counts['image']} | 🎞️ Videos: {counts['video']} | 🔊 Audio: {counts['audio']}")

    return dataset_summary


In [ ]:
# ==============================
#ASVspoof_data_preprocesssing
# ==============================
import os
import random
import torch
import torchaudio
import torchaudio.functional as F
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==============================
# CONFIGURATION
# ==============================
DATASET_BASE = Path("datasets/ASVspoof")
OUTPUT_BASE = Path("preprocessed_asvspoof")

SUBSETS = [
    "ASVspoof2019_LA_train",
    "ASVspoof2019_LA_dev",
    "ASVspoof2019_LA_eval",
]

TARGET_SR = 16000         # target sample rate for model input
MAX_WORKERS = min(8, os.cpu_count() or 4)
AUGMENT = True
VERBOSE = True

# ==============================
# AUDIO AUGMENTATION FUNCTIONS
# ==============================
def augment_audio(audio, sr):
    """Applies small random augmentations on waveform tensor."""
    # Random pitch shift
    if random.random() < 0.5:
        n_steps = random.uniform(-2, 2)
        audio = F.pitch_shift(audio, sample_rate=sr, n_steps=n_steps)

    # Random additive noise
    if random.random() < 0.4:
        noise = torch.randn_like(audio) * random.uniform(0.001, 0.005)
        audio = audio + noise

    # Random gain
    if random.random() < 0.4:
        gain_db = random.uniform(-3, 3)
        audio = F.gain(audio, gain_db)

    # Clamp
    audio = torch.clamp(audio, -1.0, 1.0)
    return audio


# ==============================
# CORE PROCESSING FUNCTION
# ==============================
def process_audio_file(audio_path, subset_name):
    try:
        audio_path = Path(audio_path)
        rel = audio_path.relative_to(DATASET_BASE / subset_name)
        out_path = OUTPUT_BASE / subset_name / rel.with_suffix(".pt")
        out_path.parent.mkdir(parents=True, exist_ok=True)

        # Load audio
        waveform, sr = torchaudio.load(audio_path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample if needed
        if sr != TARGET_SR:
            waveform = F.resample(waveform, sr, TARGET_SR)
            sr = TARGET_SR

        # Optional augmentations
        if AUGMENT and "train" in subset_name.lower():
            waveform = augment_audio(waveform, sr)

        # Normalize amplitude
        waveform = waveform / (waveform.abs().max() + 1e-6)

        # Compute Mel spectrogram for CNN+Transformer
        mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=512,
            hop_length=160,
            n_mels=80,
        )(waveform)
        mel_db = torchaudio.transforms.AmplitudeToDB()(mel_spec)

        # Save waveform and spectrogram
        torch.save(
            {
                "waveform": waveform,
                "mel_spec": mel_db,
                "sample_rate": sr,
            },
            out_path,
        )

        if VERBOSE:
            print(f"✅ {subset_name}: {rel} → saved {out_path.name}")

        return 1

    except Exception as e:
        print(f"[ERR] {audio_path}: {e}")
        return 0


# ==============================
# PARALLEL EXECUTION
# ==============================
def find_all_audio_files(subset_dir):
    files = []
    for ext in ("*.flac", "*.wav"):
        files.extend(sorted((subset_dir / "flac").rglob(ext)))
    return files


def run_parallel(subsets, workers=MAX_WORKERS):
    jobs = []
    for subset in subsets:
        subset_dir = DATASET_BASE / subset
        if not subset_dir.exists():
            continue
        files = find_all_audio_files(subset_dir)
        for f in files:
            jobs.append((f, subset))

    print(f"\n🎧 Found {len(jobs)} audio files across {len(subsets)} subsets.")
    print(f"Using {workers} workers for preprocessing...\n")

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(process_audio_file, path, ds) for path, ds in jobs]
        for _ in tqdm(as_completed(futures), total=len(futures)):
            pass

    print("\n✅ All audio files processed successfully!")


# ==============================
# MAIN
# ==============================
if __name__ == "__main__":
    random.seed(42)
    torch.manual_seed(42)
    np.random.seed(42)

    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

    print("Settings:")
    print(f" TARGET_SR     : {TARGET_SR}")
    print(f" AUGMENT       : {AUGMENT}")
    print(f" MAX_WORKERS   : {MAX_WORKERS}")
    print(f" OUTPUT_BASE   : {OUTPUT_BASE}")
    print("-" * 50)

    run_parallel(SUBSETS, workers=MAX_WORKERS)


In [ ]:
DATASET_BASE = "datasets"
OUTPUT_BASE = "preprocessed"

video_datasets = ["FaceForensics++", "CelebDF-v2"]

for dataset_name in video_datasets:
    dataset_path = os.path.join(DATASET_BASE, dataset_name)
    print(f"\n🎞 Processing {dataset_name}")

    for root, _, files in os.walk(dataset_path):
        for file in tqdm(files, desc=f"{dataset_name}"):
            if file.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
                video_path = os.path.join(root, file)
                relative = os.path.relpath(root, dataset_path)
                output_dir = os.path.join(OUTPUT_BASE, dataset_name, relative, os.path.splitext(file)[0])
                extract_and_preprocess_video(video_path, output_dir, frame_rate=5, size=(224, 224), face_crop=True)


In [2]:
# ==============================
#Visual_data_preprocessing
# ==============================

import os
from tqdm import tqdm

DATASET_BASE = "./datasets/FaceForensics++"
OUTPUT_BASE = "./processed_data/FaceForensics++"
os.makedirs(OUTPUT_BASE, exist_ok=True)

def process_faceforensics(dataset_type):
    dataset_path = os.path.join(DATASET_BASE, dataset_type)
    for root, _, files in os.walk(dataset_path):
        for f in tqdm(files, desc=f"{dataset_type}"):
            fpath = os.path.join(root, f)
            fname = os.path.splitext(f)[0]
            out_dir = os.path.join(OUTPUT_BASE, dataset_type, fname)
            os.makedirs(out_dir, exist_ok=True)
            preprocess_input(fpath, out_dir)

# original (real videos)
process_faceforensics("original_sequences/youtube/c40/videos")
process_faceforensics("original_sequences/actors/c40/videos")

# manipulated (fake videos)
manipulated_folders = [
    "manipulated_sequences/DeepFakeDetection/c40/videos",
    "manipulated_sequences/Deepfakes/c40/videos",
    "manipulated_sequences/Face2Face/c40/videos",
    "manipulated_sequences/FaceShifter/c40/videos",
    "manipulated_sequences/FaceSwap/c40/videos",
    "manipulated_sequences/NeuralTextures/c40/videos"
]
for folder in manipulated_folders:
    process_faceforensics(folder)


In [ ]:
sample_img = preprocess_image("datasets/FaceForensics++/original_sequences/actors/img001.jpg")
plt.imshow(sample_img)
plt.title("Preprocessed Image")
plt.axis("off")
plt.show()


In [2]:
# video_preprocess_fast.py
# Two-stage fast preprocessor:
# 1) ffmpeg -> sample frames per video
# 2) batch face detection (MTCNN GPU) -> crop -> augment -> save

import os
import sys
import math
import shutil
import random
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from PIL import Image, ImageEnhance

from tqdm import tqdm

# try GPU MTCNN
USE_MTCNN = True
try:
    from facenet_pytorch import MTCNN
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mtcnn = MTCNN(keep_all=False, device=device, post_process=False)
except Exception:
    USE_MTCNN = False
    mtcnn = None

# fallback Haar
import cv2
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# ----------------- USER SETTINGS -----------------
DATASET_BASE = Path("datasets")
OUTPUT_BASE = Path("preprocessed_asvspoof")
VIDEO_DATASETS = ["FaceForensics++", "CelebDF-v2"]   # adjust if needed

TMP_FRAMES = Path("tmp_frames")   # temp frames root
TMP_FRAMES.mkdir(exist_ok=True)

TARGET_SIZE = (224, 224)
WORKERS = min(6, os.cpu_count() or 4)   # parallel videos
BATCH_DETECT = 64    # how many frames to detect at once on GPU
FPS_SHORT = 3        # frames/sec for short clips
FPS_MED = 2          # medium
FPS_LONG = 1         # long
SHORT_SECONDS = 8
MED_SECONDS = 60

AUGMENT = True
MIN_FACE_SIDE = 64   # minimum crop side to accept

# small, very fast augmentation (PIL)
def augment_pil(img_pil):
    if random.random() < 0.5:
        img_pil = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.35:
        enhancer = ImageEnhance.Brightness(img_pil)
        img_pil = enhancer.enhance(random.uniform(0.92, 1.08))
    if random.random() < 0.25:
        enhancer = ImageEnhance.Contrast(img_pil)
        img_pil = enhancer.enhance(random.uniform(0.92, 1.08))
    return img_pil

# choose fps per video length
def choose_fps(duration_seconds):
    if duration_seconds <= SHORT_SECONDS:
        return FPS_SHORT
    if duration_seconds <= MED_SECONDS:
        return FPS_MED
    return FPS_LONG

# stage 1: fast frame extraction via ffmpeg
def dump_frames_ffmpeg(video_path: Path, out_dir: Path, fps: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    # ffmpeg command:
    # - hide banner, overwrite, set fps, output sequential jpg
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-i", str(video_path),
        "-vf", f"fps={fps}",
        "-qscale:v", "2",
        str(out_dir / "frame_%06d.jpg")
    ]
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as e:
        # fail quietly, return False
        print(f"[FFMPEG ERR] {video_path}: {e}")
        return False

# helper: read PIL image safe
def pil_open(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

# stage 2: batch detect and crop
def batch_detect_and_save(frame_paths, save_dir: Path, prefix="", use_mtcnn=USE_MTCNN):
    imgs = []
    valid_paths = []
    for p in frame_paths:
        img = pil_open(p)
        if img is None:
            continue
        imgs.append(img)
        valid_paths.append(p)
    if len(imgs) == 0:
        return 0
    saved = 0
    # run MTCNN in batch
    if use_mtcnn and mtcnn is not None:
        try:
            # facenet-pytorch accepts list of PIL images and returns boxes as numpy array
            boxes_batch, _ = mtcnn.detect(imgs)
        except Exception as e:
            # if mtcnn fails, fallback to haar per-image
            boxes_batch = [None] * len(imgs)
    else:
        boxes_batch = [None] * len(imgs)
    for i, img in enumerate(imgs):
        boxes = None
        if boxes_batch is not None:
            boxes = boxes_batch[i]
        # fallback to Haar if no MTCNN box
        if boxes is None or len(boxes) == 0:
            # use grayscale Haar detection
            npim = np.array(img)
            gray = cv2.cvtColor(npim, cv2.COLOR_RGB2GRAY)
            try:
                faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.08, minNeighbors=4, minSize=(48,48))
            except cv2.error:
                faces = []
            if len(faces) > 0:
                x,y,w,h = max(faces, key=lambda r: r[2]*r[3])
                boxes = np.array([[x, y, x+w, y+h]])
            else:
                boxes = None
        if boxes is None or len(boxes) == 0:
            # save whole resized frame as fallback
            out = img.resize(TARGET_SIZE, Image.BICUBIC)
            if AUGMENT:
                out = augment_pil(out)
            out.save(save_dir / f"{prefix}_full_{i:05d}.jpg", quality=95)
            saved += 1
            continue
        # if there is box array
        x1, y1, x2, y2 = boxes[0].astype(int)
        # pad slightly
        w = x2 - x1
        h = y2 - y1
        pad = int(0.18 * max(w, h))
        x0 = max(0, x1 - pad)
        y0 = max(0, y1 - pad)
        x3 = min(img.width, x2 + pad)
        y3 = min(img.height, y2 + pad)
        # crop and validate
        try:
            crop = img.crop((x0, y0, x3, y3))
        except Exception:
            crop = img
        if crop.width < MIN_FACE_SIDE or crop.height < MIN_FACE_SIDE:
            # fallback to whole frame
            out = img.resize(TARGET_SIZE, Image.BICUBIC)
            if AUGMENT:
                out = augment_pil(out)
            out.save(save_dir / f"{prefix}_full_{i:05d}.jpg", quality=95)
            saved += 1
            continue
        out = crop.resize(TARGET_SIZE, Image.BICUBIC)
        if AUGMENT:
            out = augment_pil(out)
        out.save(save_dir / f"{prefix}_face_{i:05d}.jpg", quality=95)
        saved += 1
    return saved

# process one video: dump frames -> detect in batches -> clean tmp
def process_one_video(video_path: str, dataset_name: str):
    vpath = Path(video_path)
    rel = vpath.relative_to(DATASET_BASE / dataset_name)
    save_dir = OUTPUT_BASE / dataset_name / rel.parent / vpath.stem
    save_dir.mkdir(parents=True, exist_ok=True)

    # make temp folder for this video
    tmp_dir = TMP_FRAMES / dataset_name / rel.parent / vpath.stem
    if tmp_dir.exists():
        # remove if leftover from earlier
        try:
            shutil.rmtree(tmp_dir)
        except Exception:
            pass
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # get duration using ffprobe (fast). fallback to heuristic if fails
    dur = None
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries",
               "format=duration", "-of", "default=noprint_wrappers=1:nokey=1",
               str(vpath)]
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip()
        dur = float(out)
    except Exception:
        dur = None

    fps = choose_fps(dur if dur is not None else 30)
    ok = dump_frames_ffmpeg(vpath, tmp_dir, fps)
    if not ok:
        # try one frame per second as fallback
        dump_frames_ffmpeg(vpath, tmp_dir, 1)

    # collect frame files
    frames = sorted(tmp_dir.glob("*.jpg"))
    if len(frames) == 0:
        # fallback: try extracting single poster frame with ffmpeg
        try:
            subprocess.run(["ffmpeg","-y","-i",str(vpath),"-vf","select='eq(n,0)'","-qscale:v","2",str(tmp_dir/"frame_000001.jpg")], check=False)
            frames = sorted(tmp_dir.glob("*.jpg"))
        except Exception:
            pass

    # batch detect
    total_saved = 0
    for i in range(0, len(frames), BATCH_DETECT):
        batch = frames[i:i+BATCH_DETECT]
        saved = batch_detect_and_save(batch, save_dir, prefix=f"{i:06d}")
        total_saved += saved

    # cleanup
    try:
        shutil.rmtree(tmp_dir)
    except Exception:
        pass

    return (str(vpath), total_saved)

# find videos under datasets
def gather_jobs(datasets):
    jobs = []
    for ds in datasets:
        base = DATASET_BASE / ds
        if not base.exists():
            continue
        for ext in ("*.mp4","*.avi","*.mov","*.mkv"):
            for p in sorted(base.rglob(ext)):
                jobs.append((str(p), ds))
    return jobs

def main():
    random.seed(42)
    jobs = gather_jobs(VIDEO_DATASETS)
    print(f"Found {len(jobs)} videos. Workers {WORKERS}. MTCNN {'ON' if USE_MTCNN else 'OFF'}.")

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = [ex.submit(process_one_video, p, ds) for (p,ds) in jobs]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            try:
                vpath, saved = fut.result()
            except Exception as e:
                print("Job failed:", e)
                continue

if __name__ == '__main__':
    main()


C:\Users\admin\anaconda3\envs\deepfake\lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict

Found 15960 videos. Workers 6. MTCNN ON.


 60%|█████████████████████████████████████████████                              | 9602/15960 [6:20:45<42:38,  2.49it/s]

[FFMPEG ERR] datasets\CelebDF-v2\Celeb-real\id27_0005.mp4: Command '['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', 'datasets\\CelebDF-v2\\Celeb-real\\id27_0005.mp4', '-vf', 'fps=3', '-qscale:v', '2', 'tmp_frames\\CelebDF-v2\\Celeb-real\\id27_0005\\frame_%06d.jpg']' returned non-zero exit status 4294967274.
[FFMPEG ERR] datasets\CelebDF-v2\Celeb-real\id27_0005.mp4: Command '['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', 'datasets\\CelebDF-v2\\Celeb-real\\id27_0005.mp4', '-vf', 'fps=1', '-qscale:v', '2', 'tmp_frames\\CelebDF-v2\\Celeb-real\\id27_0005\\frame_%06d.jpg']' returned non-zero exit status 4294967274.


100%|██████████████████████████████████████████████████████████████████████████| 15960/15960 [8:05:48<00:00,  1.83s/it]


In [3]:
import os
import csv
from pathlib import Path

DATA_ROOT = Path("preprocessed_asvspoof")
OUTPUT_CSV = "train_metadata.csv"

# Define rules
real_folders = [
    "FaceForensics++/original_sequences/actors/c40/videos",
    "FaceForensics++/original_sequences/youtube/c40/videos",
    "CelebDF-v2/YouTube-real",
    "CelebDF-v2/Celeb-real"
]

fake_folders = [
    "FaceForensics++/manipulated_sequences/DeepFakeDetection/c40/videos",
    "FaceForensics++/manipulated_sequences/Deepfakes/c40/videos",
    "FaceForensics++/manipulated_sequences/FaceShifter/c40/videos",
    "FaceForensics++/manipulated_sequences/FaceSwap/c40/videos",
    "FaceForensics++/manipulated_sequences/NeuralTextures/c40/videos",
    "CelebDF-v2/Celeb-synthesis"
]

# Helper function
def collect_images(folder, label):
    all_paths = []
    folder_path = DATA_ROOT / folder
    if not folder_path.exists():
        print(f"⚠️ Missing folder: {folder}")
        return []
    for img in folder_path.rglob("*.jpg"):
        all_paths.append((str(img), label))
    return all_paths

# Collect data
rows = []
for folder in real_folders:
    rows.extend(collect_images(folder, 0))
for folder in fake_folders:
    rows.extend(collect_images(folder, 1))

print(f"✅ Total images collected: {len(rows):,}")

# Save CSV
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "label"])
    writer.writerows(rows)

print(f"📄 CSV saved as: {OUTPUT_CSV}")


✅ Total images collected: 544,876
📄 CSV saved as: train_metadata.csv


In [8]:
with open("train_metadata.csv", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())


path,label
preprocessed_asvspoof\FaceForensics++\original_sequences\actors\c40\videos\01__exit_phone_room\000000_face_00000.jpg,0
preprocessed_asvspoof\FaceForensics++\original_sequences\actors\c40\videos\01__exit_phone_room\000000_face_00001.jpg,0
preprocessed_asvspoof\FaceForensics++\original_sequences\actors\c40\videos\01__exit_phone_room\000000_face_00002.jpg,0
preprocessed_asvspoof\FaceForensics++\original_sequences\actors\c40\videos\01__exit_phone_room\000000_face_00004.jpg,0


In [9]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import torchvision.transforms as transforms

# ======================
# CONFIG
# ======================
CSV_PATH = "train_metadata.csv"  # your generated metadata file
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_WORKERS = 4  # increase if GPU + CPU are strong

# ======================
# 1. LOAD CSV
# ======================
# (Your CSV uses commas, not tabs)
df = pd.read_csv(CSV_PATH)

# Clean up bad rows
df = df.dropna(subset=["path", "label"])
df["path"] = df["path"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

# Filter out empty labels and non-integer values
df = df[df["label"].isin(["0", "1"])]
df["label"] = df["label"].astype(int)

print(f"✅ Cleaned dataset: {len(df)} samples")
print(df.head())

✅ Cleaned dataset: 544876 samples
                                                path  label
0  preprocessed_asvspoof\FaceForensics++\original...      0
1  preprocessed_asvspoof\FaceForensics++\original...      0
2  preprocessed_asvspoof\FaceForensics++\original...      0
3  preprocessed_asvspoof\FaceForensics++\original...      0
4  preprocessed_asvspoof\FaceForensics++\original...      0


In [10]:


# ======================
# 2. SPLIT DATA
# ======================
train_df, test_df = train_test_split(
    df, test_size=0.1, stratify=df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label"], random_state=42
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ======================
# 3. DEFINE DATASET CLASS
# ======================
class DeepfakeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])

        # Handle missing or corrupted images gracefully
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Skipping broken file: {img_path}")
            # Return a blank image with label 0 (or skip in collate_fn)
            image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# ======================
# 4. TRANSFORMS
# ======================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======================
# 5. DATALOADERS
# ======================
train_dataset = DeepfakeDataset(train_df, transform=train_transform)
val_dataset = DeepfakeDataset(val_df, transform=val_transform)
test_dataset = DeepfakeDataset(test_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("✅ Dataloaders ready!")
print(f"Total train batches: {len(train_loader)}")
print(f"Total val batches: {len(val_loader)}")
print(f"Total test batches: {len(test_loader)}")


Train: 441349 | Val: 49039 | Test: 54488
✅ Dataloaders ready!
Total train batches: 13793
Total val batches: 1533
Total test batches: 1703


In [ ]:
# final_training_with_amp_and_tensorboard.py
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import timm
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm


# CONFIG
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 50
BATCH_EFFECTIVE = None  # not used here, kept for future gradient accumulation
LR = 1e-4
NUM_CLASSES = 2
SAVE_DIR = "checkpoints"
LOG_DIR = "runs/training_" + time.strftime("%Y%m%d-%H%M%S")
SAVE_PERIOD_EPOCHS = 5      # save full checkpoint every N epochs (in addition to best)
BEST_CKPT = os.path.join(SAVE_DIR, "best_model.pth")
LAST_CKPT = os.path.join(SAVE_DIR, "last_model.pth")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Save dir: {SAVE_DIR}")
print(f"TensorBoard logs: {LOG_DIR}")

# MODEL: EfficientNet-B4 + ViT-base mid-level fusion
class CNN_Transformer_Fusion(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # EfficientNet-B4 as CNN backbone
        self.cnn = models.efficientnet_b4(weights="IMAGENET1K_V1")
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792

        # ViT (head removed)
        self.transformer = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.transformer.head = nn.Identity()
        trans_out = 768

        # small classifier after concatenation
        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + trans_out, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # x: [B, 3, H, W]
        cnn_feat = self.cnn(x)          # [B, cnn_out]
        trans_feat = self.transformer(x) # [B, trans_out]
        fused = torch.cat([cnn_feat, trans_feat], dim=1)
        return self.fusion(fused)

# Create model, criterion, optimizer, scheduler, scaler
model = CNN_Transformer_Fusion(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

# TensorBoard writer
writer = SummaryWriter(LOG_DIR)

# Resume support
start_epoch = 0
best_val_acc = 0.0
if os.path.exists(BEST_CKPT):
    try:
        ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt.get("scheduler", scheduler.state_dict()))
        scaler.load_state_dict(ckpt.get("scaler", scaler.state_dict()))
        best_val_acc = ckpt.get("best_acc", 0.0)
        start_epoch = ckpt.get("epoch", 0) + 1
        print(f"Resumed best checkpoint at epoch {start_epoch-1} with acc {best_val_acc:.4f}")
    except Exception as e:
        print("Could not fully resume best checkpoint:", e)

if os.path.exists(LAST_CKPT):
    try:
        ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
        # only resume last if start_epoch remains 0 (prefers best)
        if start_epoch == 0:
            model.load_state_dict(ckpt["model"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scheduler.load_state_dict(ckpt.get("scheduler", scheduler.state_dict()))
            scaler.load_state_dict(ckpt.get("scaler", scaler.state_dict()))
            start_epoch = ckpt.get("epoch", 0) + 1
            best_val_acc = ckpt.get("best_acc", best_val_acc)
            print(f"Resumed last checkpoint at epoch {start_epoch-1}")
    except Exception as e:
        print("Could not resume last checkpoint:", e)

# Sanity check: ensure dataloaders exist
try:
    _ = iter(train_loader)
except NameError:
    raise RuntimeError("train_loader not found. Define train_loader, val_loader, test_loader before running this script.")

# Utility: evaluate on loader
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Eval", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            loss_sum += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return loss_sum / max(1, total), correct / max(1, total)



In [ ]:
# Training loop with AMP, logging, checkpointing
try:
    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Train", leave=False)
        for imgs, labels in loop:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # metrics
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)

        # epoch train metrics
        train_loss = running_loss / max(1, running_total)
        train_acc = running_correct / max(1, running_total)

        # validation
        val_loss, val_acc = evaluate(model, val_loader)

        # scheduler step (after validation is OK)
        scheduler.step()

        # Logging to console and TensorBoard
        print(f"Epoch {epoch+1} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f} | Time {(time.time()-t0):.1f}s")
        writer.add_scalar("Loss/train", train_loss, epoch+1)
        writer.add_scalar("Loss/val", val_loss, epoch+1)
        writer.add_scalar("Acc/train", train_acc, epoch+1)
        writer.add_scalar("Acc/val", val_acc, epoch+1)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch+1)

        # Save last checkpoint
        last_state = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_acc": best_val_acc
        }
        torch.save(last_state, LAST_CKPT)

        # Save best checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = dict(last_state)
            best_state["best_acc"] = best_val_acc
            torch.save(best_state, BEST_CKPT)
            print(f"Saved new best model at epoch {epoch+1} (val_acc={best_val_acc:.4f})")

        # Periodic full save
        if (epoch + 1) % SAVE_PERIOD_EPOCHS == 0:
            periodic_path = os.path.join(SAVE_DIR, f"ckpt_epoch_{epoch+1}.pth")
            torch.save(last_state, periodic_path)
            print(f"Saved periodic checkpoint: {periodic_path}")

    # After training, run final test evaluation once
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"\nFinal Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
    writer.add_scalar("Loss/test", test_loss, epoch+1)
    writer.add_scalar("Acc/test", test_acc, epoch+1)

except KeyboardInterrupt:
    print("\nTraining interrupted by user. Saving last checkpoint...")
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_acc": best_val_acc
    }, LAST_CKPT)
    print("Saved last checkpoint. You can resume later.")
finally:
    writer.close()


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm
import timm
from torch.cuda.amp import GradScaler, autocast

# ======================
# CONFIG
# ======================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 2
SAVE_DIR = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"🔥 Training on {DEVICE}")

# ======================
# MODEL
# ======================
class CNN_Transformer_Fusion(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # --- CNN Backbone ---
        self.cnn = models.efficientnet_b4(weights="IMAGENET1K_V1")
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792  # EfficientNet-B4 output dim

        # --- Transformer Backbone ---
        self.transformer = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.transformer.head = nn.Identity()
        trans_out = 768

        # --- Fusion ---
        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + trans_out, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)
        trans_feat = self.transformer(x)
        fused = torch.cat([cnn_feat, trans_feat], dim=1)
        return self.fusion(fused)

# ======================
# TRAIN + VALIDATION
# ======================
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(loader, desc="Training", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate(model, loader, criterion):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validating", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return val_loss / total, correct / total

# ======================
# INITIALIZE EVERYTHING
# ======================
model = CNN_Transformer_Fusion(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

# --- Resume if checkpoint exists ---
start_epoch = 0
best_val_acc = 0.0
ckpt_path = os.path.join(SAVE_DIR, "best_model.pth")

if os.path.exists(ckpt_path):
    print("🔁 Resuming from checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    best_val_acc = checkpoint["best_acc"]
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed from epoch {start_epoch} (Best acc: {best_val_acc:.4f})")

# ======================
# TRAINING LOOP
# ======================
for epoch in range(start_epoch, EPOCHS):
    print(f"\n🚀 Epoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # --- Save checkpoint ---
    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_acc": best_val_acc
    }

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(state, ckpt_path)
        print(f"✅ New best model saved (Acc: {best_val_acc:.4f})")

print("\n🏁 Training complete!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
